In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../train.csv")

In [17]:
df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"]
df["commits_per_active_day"].isna().sum()

np.int64(0)

In [25]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_log_error

y = df["hours_spent"] # target
y_log = np.log1p(y)
X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"]).copy()
for col in X.select_dtypes(include=["object", "string"]).columns:
        X[col] = X[col].astype("category")

use_categories = True
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X))                 
for train_idx, val_idx in skf.split(X, df["hours_bucket"]):
    kw = {"categorical_features": "from_dtype"} if use_categories else {}
    model = HistGradientBoostingRegressor(**kw)
    model.fit(X.iloc[train_idx], y_log.iloc[train_idx]) 
    oof[val_idx] = model.predict(X.iloc[val_idx])

rmsle = root_mean_squared_log_error(y, np.clip(np.expm1(oof), 0, None))
print(rmsle)



0.6689300359598184


In [19]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [16]:
import joblib

In [13]:
joblib.dump(model, "../model.joblib")

['../model.joblib']